<a href="https://colab.research.google.com/github/cwf2/ccc2026/blob/main/6%20-%20PCA%20vs%20dialogism%20gap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Where PCA and dialogism disagree

The two speechiness methods agree strongly overall (`4 - Compare speechiness methods` found Pearson r ≈ 0.87–0.89 across window sizes) — but not everywhere. This notebook has two goals:

1. **Find the regions where they diverge most**, so we can zoom into the actual tokens driving the disagreement, using the same viewing-window/text-display approach as `5 - Interactive`.
2. **Test whether Nonnus is a special case.** Burns' dialogism method ranks lexical/grammatical features by how well they distinguish speech from narrative *pooled across all six authors at once*. Our PCA + logistic regression method trains on samples balanced across (author, class) pairs and projects onto principal components that can absorb some author-specific stylistic variation before the speech/narrative boundary is drawn. If Nonnus' narrative style is distinctive enough to read as "speechy" under a pooled, author-blind lexicon — as Burns' method would predict — but PCA's more author-aware boundary sees through that, we'd expect Nonnus' narration to score elevated under dialogism specifically, more than it does under PCA.

In [ ]:
# only install if ccc2026 isn't already importable (e.g. fresh Colab), so
# running this cell doesn't clobber a local editable dev install every time
try:
    import ccc2026
except ImportError:
    %pip install git+https://github.com/cwf2/ccc2026

### Import statements

In [ ]:
import ccc2026
from ccc2026 import dialogism, viz

import pandas as pd
import seaborn as sns
from IPython.display import display, HTML
from matplotlib import pyplot as plt

ccc2026.setup()

### Train once, at one window size

Same fixed sample size/seed as `5 - Interactive`, and the same window (`200`) `4 - Compare speechiness methods` settled on for its own cross-method comparisons — kept consistent so this notebook's numbers are comparable to that one. Goal 2's robustness check below reruns at a few other window sizes to confirm the finding isn't an artifact of this particular choice.

In [ ]:
PCA_SAMPLE_SIZE = 1000
PCA_SEED = 1
WINDOW = 200

feature_set = {
    "lemma": ccc2026.top_lemmas,
    "pos": ccc2026.all_pos,
    "morph": ccc2026.top_morph,
}
train = ccc2026.run_training(feature_set, sample_size=PCA_SAMPLE_SIZE, seed=PCA_SEED)

lexicons = dialogism.build_lexicons()
dialogism_score = dialogism.token_dialogism_score(lexicons)

print("Trained.")

### Rolling scores, z-scored corpus-wide

Both methods' rolling scores, standardized against the *whole corpus* (not per-book, unlike `viz.plot_overlay`) — comparing authors against each other requires one shared baseline, not six book-relative ones. `gap = z_pca - z_dialogism`: positive means PCA rates a window more speech-like than dialogism does, relative to how each method rates the corpus as a whole.

In [ ]:
pca_roll = ccc2026.rolling_samples(train, window_size=WINDOW)["speech_score"]["score"]
dialogism_roll = dialogism.rolling_dialogism(dialogism_score, window_size=WINDOW)["speech_score"]["score"]

gap_df = pd.DataFrame({"pca": pca_roll, "dialogism": dialogism_roll}).dropna()
gap_df["z_pca"] = (gap_df["pca"] - gap_df["pca"].mean()) / gap_df["pca"].std()
gap_df["z_dialogism"] = (gap_df["dialogism"] - gap_df["dialogism"].mean()) / gap_df["dialogism"].std()
gap_df["gap"] = gap_df["z_pca"] - gap_df["z_dialogism"]
gap_df["abs_gap"] = gap_df["gap"].abs()

gap_df["work"] = ccc2026.tokens.loc[gap_df.index, "work"]
gap_df["pref"] = ccc2026.tokens.loc[gap_df.index, "pref"]
gap_df["line"] = ccc2026.tokens.loc[gap_df.index, "line"]
gap_df["speaker"] = ccc2026.tokens.loc[gap_df.index, "speaker"]

label = pd.Series("speech", index=gap_df.index)
label[gap_df["speaker"] == "Odysseus-Apologue"] = "other"
label[gap_df["speaker"].isna()] = "narration"
gap_df["label"] = label

print(f"{len(gap_df)} windows")
gap_df[["z_pca", "z_dialogism", "gap"]].describe()

## Goal 1: regions of biggest disagreement

Top windows by `abs_gap`, with non-max suppression so the list isn't dominated by many overlapping windows from the same passage (a real disagreement tends to span several adjacent window-centers, not just one).

In [ ]:
def top_divergent_regions(gap_df, n=15, min_index_gap=300, top_pool=3000):
    '''Top n windows by |gap|, skipping any candidate within min_index_gap
    token-positions of an already-picked window in the same book — keeps the
    list from being dominated by one wide disagreement in a single passage.
    '''
    pool = gap_df.sort_values("abs_gap", ascending=False).head(top_pool)
    picked_idx, rows = [], []
    for idx, row in pool.iterrows():
        too_close = any(
            row["work"] == r["work"] and row["pref"] == r["pref"] and abs(idx - pi) < min_index_gap
            for pi, r in zip(picked_idx, rows)
        )
        if not too_close:
            picked_idx.append(idx)
            rows.append(row)
        if len(rows) >= n:
            break
    return gap_df.loc[picked_idx]

top_regions = top_divergent_regions(gap_df, n=15)
top_regions[["work", "pref", "line", "z_pca", "z_dialogism", "gap", "speaker"]]

### Drill into the top few

For each of the top divergent regions, the standardized signal plot (±25 lines) plus the highlighted text — same views as `5 - Interactive`, built directly here rather than requiring the widgets.

In [ ]:
N_DRILLDOWN = 5
LINE_MARGIN = 25

for idx, region in top_regions.head(N_DRILLDOWN).iterrows():
    work, pref, line = region["work"], region["pref"], int(region["line"])
    first_line, last_line = max(line - LINE_MARGIN, 1), line + LINE_MARGIN

    print(f"{work} {pref}, around line {line}  (gap={region['gap']:.2f}, "
          f"z_pca={region['z_pca']:.2f}, z_dialogism={region['z_dialogism']:.2f}, "
          f"speaker={region['speaker']})")

    fig = viz.plot_overlay(ccc2026.tokens, work, pref, pca_roll, dialogism_roll,
                            first_line=first_line, last_line=last_line)
    display(fig)
    plt.close(fig)

    mask = (ccc2026.tokens["work"] == work) & (ccc2026.tokens["pref"] == pref)
    book_tokens = ccc2026.tokens.loc[mask]
    display_col = viz.build_display_column(book_tokens, lexicons)
    ccc2026.tokens.loc[book_tokens.index, "display"] = display_col
    html = viz.highlighted_excerpt(ccc2026.tokens, work, pref, first_line=first_line, last_line=last_line)
    display(HTML(html))

## Goal 2: does Nonnus diverge more?

Restricting to windows centered on narration tokens only (`speaker` is null — same definition used throughout the package, excluding Odysseus' Apologue as its own "other" category). For each work, the mean z-scored narration score under each method — both on the same corpus-wide baseline computed above, so the means are directly comparable across authors and between methods.

If Burns' prediction holds and PCA is comparatively blind to it, Dionysiaca's narration should sit noticeably higher (more speech-like) under `z_dialogism` than under `z_pca`, relative to the other five works.

In [ ]:
narration = gap_df[gap_df["label"] == "narration"]

by_work = narration.groupby("work")[["z_pca", "z_dialogism"]].agg(["mean", "sem"])
by_work.columns = ["_".join(c) for c in by_work.columns]
by_work = by_work.sort_values("z_dialogism_mean")
by_work

In [ ]:
plot_df = narration.melt(
    id_vars=["work"], value_vars=["z_pca", "z_dialogism"],
    var_name="method", value_name="z_score",
)
work_order = by_work.index.tolist()

g = sns.catplot(
    data=plot_df, x="work", y="z_score", hue="method",
    order=work_order, kind="bar", errorbar="se", height=4.5, aspect=1.6,
)
g.set(
    title=f"Narration-only score by author (window={WINDOW})",
    xlabel="", ylabel="z-score (corpus-wide baseline)",
)
g.ax.axhline(0, color="k", ls="--", lw=1)
plt.xticks(rotation=20, ha="right")
plt.show()

### Robustness check across window sizes

Same per-work narration means, recomputed at a few other window sizes, to confirm the ranking isn't an artifact of `WINDOW=200` specifically.

In [ ]:
robustness_rows = []
for w in [50, 200, 500, 1000]:
    p_roll = ccc2026.rolling_samples(train, window_size=w)["speech_score"]["score"]
    d_roll = dialogism.rolling_dialogism(dialogism_score, window_size=w)["speech_score"]["score"]
    both_w = pd.DataFrame({"pca": p_roll, "dialogism": d_roll}).dropna()
    both_w["z_pca"] = (both_w["pca"] - both_w["pca"].mean()) / both_w["pca"].std()
    both_w["z_dialogism"] = (both_w["dialogism"] - both_w["dialogism"].mean()) / both_w["dialogism"].std()
    both_w["work"] = ccc2026.tokens.loc[both_w.index, "work"]
    both_w["speaker"] = ccc2026.tokens.loc[both_w.index, "speaker"]
    narr_w = both_w[both_w["speaker"].isna()]
    means = narr_w.groupby("work")[["z_pca", "z_dialogism"]].mean()
    means["window"] = w
    robustness_rows.append(means)

robustness = pd.concat(robustness_rows).reset_index().rename(columns={"index": "work"})

g = sns.relplot(
    data=robustness.melt(id_vars=["work", "window"], value_vars=["z_pca", "z_dialogism"],
                          var_name="method", value_name="z_score"),
    x="window", y="z_score", hue="work", style="method", kind="line",
    markers=True, height=4.5, aspect=1.6,
)
g.set(title="Narration z-score by author, across window sizes", xlabel="window size", ylabel="z-score")
g.ax.axhline(0, color="k", ls="--", lw=1)
plt.show()

robustness.pivot(index="work", columns="window", values=["z_pca", "z_dialogism"]).round(2)

### What we actually see

Run the cells above and compare the numbers/plots to the prediction in the intro before drawing conclusions here — this section is deliberately left for interpretation after the fact rather than asserted in advance.